# Prepare Human Validation Annotation Sheet

This notebook creates a user-friendly Excel workbook for human validation of LLM relevance labels. It samples 40 queries, exports all 50 blinded candidate recipes for each sampled query, and adds spreadsheet controls so annotators can only enter valid relevance labels.

The notebook does not show LLM labels, retrieval methods, model scores, RRF ranks, or any other signal that could bias the annotator.


## 1. Setup

The output workbook contains:

- `rubric`: relevance definitions and annotation instructions.
- `annotation`: one row per query-document pair with blank `human_relevance` and `human_notes` columns.
- `sampled_queries`: the sampled query list for audit.

The annotator should fill only `human_relevance` and optionally `human_notes`.


In [1]:
from __future__ import annotations

import json
import math
import random
from pathlib import Path
from typing import Optional

import pandas as pd

try:
    from openpyxl import load_workbook
    from openpyxl.comments import Comment
    from openpyxl.formatting.rule import FormulaRule
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.utils import get_column_letter
    from openpyxl.worksheet.datavalidation import DataValidation
except ImportError as import_error:
    raise ImportError(
        "openpyxl is required to create the Excel annotation sheet. "
        "Install it with `pip install openpyxl` in the notebook environment."
    ) from import_error


def find_finalproject_root(start_path: Optional[Path] = None) -> Path:
    """Find the Finalproject directory by walking upward from the current path."""
    current_path = (start_path or Path.cwd()).resolve()
    for candidate_path in [current_path, *current_path.parents]:
        if candidate_path.name == "Finalproject":
            return candidate_path
        nested_finalproject = candidate_path / "Finalproject"
        if nested_finalproject.exists() and nested_finalproject.is_dir():
            return nested_finalproject.resolve()
    raise FileNotFoundError("Could not find the Finalproject directory.")


FINALPROJECT_ROOT = find_finalproject_root()
NOTEBOOK_OUTPUT_DIR = FINALPROJECT_ROOT / "notebooks" / "rec_and_eval" / "groundtruth_outputs"
ANNOTATION_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "annotation"
HUMAN_VALIDATION_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "human_validation"

BLINDED_ANNOTATION_ITEMS_PATH = ANNOTATION_OUTPUT_DIR / "blinded_annotation_items.jsonl"
QUERY_SET_PATH = NOTEBOOK_OUTPUT_DIR / "queries_500_title_as_query.csv"

HUMAN_VALIDATION_QUERY_COUNT = 40
HUMAN_VALIDATION_RANDOM_SEED = 20260713

ANNOTATION_WORKBOOK_PATH = HUMAN_VALIDATION_OUTPUT_DIR / "human_validation_annotation_sheet.xlsx"
COMPLETED_WORKBOOK_PATH = HUMAN_VALIDATION_OUTPUT_DIR / "human_validation_annotation_sheet_completed.xlsx"
COMPLETED_CSV_PATH = HUMAN_VALIDATION_OUTPUT_DIR / "human_validation_annotation_sheet_completed.csv"

print("Finalproject root:", FINALPROJECT_ROOT)
print("Blinded annotation items:", BLINDED_ANNOTATION_ITEMS_PATH)
print("Query set:", QUERY_SET_PATH)
print("Workbook output:", ANNOTATION_WORKBOOK_PATH)


Finalproject root: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject
Blinded annotation items: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\annotation\blinded_annotation_items.jsonl
Query set: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\queries_500_title_as_query.csv
Workbook output: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\human_validation\human_validation_annotation_sheet.xlsx


## 2. Load Blinded Annotation Items

This section loads the already shuffled/blinded candidate pool. Each query should have 50 candidate documents. The annotator-facing workbook uses only query text and recipe metadata; it does not include LLM labels or retrieval scores.


In [2]:
def load_jsonl_records(jsonl_path: Path) -> list[dict]:
    """Load a JSONL file into a list of dictionaries."""
    records = []
    with jsonl_path.open("r", encoding="utf-8") as input_file:
        for line_number, line in enumerate(input_file, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON at {jsonl_path}:{line_number}") from error
    return records


def build_candidate_id(blinded_position: int) -> str:
    """Build the same short candidate ID used by the LLM prompt."""
    return f"D{int(blinded_position):03d}"


def compact_text(value: object, max_characters: int = 900) -> str:
    """Convert text to a compact one-cell string."""
    text = str(value or "").strip()
    if len(text) > max_characters:
        return text[: max_characters - 3].rstrip() + "..."
    return text


blinded_items = load_jsonl_records(BLINDED_ANNOTATION_ITEMS_PATH)
blinded_dataframe = pd.DataFrame(blinded_items)
required_columns = {
    "query_id",
    "query_text",
    "doc_id",
    "blinded_position",
    "recipe_title",
    "recipe_type",
    "recipe_description",
}
missing_columns = required_columns - set(blinded_dataframe.columns)
if missing_columns:
    raise ValueError(f"Blinded annotation items are missing required columns: {sorted(missing_columns)}")

blinded_dataframe["query_id"] = blinded_dataframe["query_id"].astype(int)
blinded_dataframe["doc_id"] = blinded_dataframe["doc_id"].astype(int)
blinded_dataframe["blinded_position"] = blinded_dataframe["blinded_position"].astype(int)
blinded_dataframe["candidate_id"] = blinded_dataframe["blinded_position"].apply(build_candidate_id)

labels_per_query = blinded_dataframe.groupby("query_id").size()
print("Blinded rows:", len(blinded_dataframe))
print("Query count:", blinded_dataframe["query_id"].nunique())
print("Candidates per query distribution:")
print(labels_per_query.value_counts().sort_index())

if not (labels_per_query == 50).all():
    raise ValueError("Every query is expected to have exactly 50 blinded candidates.")


Blinded rows: 25000
Query count: 500
Candidates per query distribution:
50    500
Name: count, dtype: int64


## 3. Sample 20 Queries

Sampling is query-level: once a query is selected, all 50 candidates for that query are included. Because the query file contains `type_of_food`, the sample is approximately stratified by recipe category with a fixed random seed. This keeps the human validation subset more representative than simple random sampling while remaining reproducible.


In [3]:
def sample_queries(
    query_dataframe: pd.DataFrame,
    sample_size: int,
    random_seed: int,
    stratify_column: str | None = "type_of_food",
) -> pd.DataFrame:
    """Sample query rows, optionally with approximate proportional stratification."""
    if sample_size > len(query_dataframe):
        raise ValueError("sample_size cannot exceed the number of available queries.")

    query_dataframe = query_dataframe.copy()
    query_dataframe["query_id"] = query_dataframe["query_id"].astype(int)

    if stratify_column not in query_dataframe.columns:
        return (
            query_dataframe.sample(n=sample_size, random_state=random_seed)
            .sort_values("query_id")
            .reset_index(drop=True)
        )

    random_generator = random.Random(random_seed)
    sampled_parts = []
    grouped = list(query_dataframe.groupby(stratify_column, dropna=False))

    for _, group in grouped:
        group_fraction = len(group) / len(query_dataframe)
        group_sample_size = max(1, round(group_fraction * sample_size))
        group_sample_size = min(group_sample_size, len(group))
        sampled_parts.append(
            group.sample(
                n=group_sample_size,
                random_state=random_generator.randint(0, 10**9),
            )
        )

    sampled_dataframe = pd.concat(sampled_parts, ignore_index=True)
    if len(sampled_dataframe) > sample_size:
        sampled_dataframe = sampled_dataframe.sample(n=sample_size, random_state=random_seed)
    elif len(sampled_dataframe) < sample_size:
        remaining_dataframe = query_dataframe.loc[
            ~query_dataframe["query_id"].isin(sampled_dataframe["query_id"])
        ]
        additional_dataframe = remaining_dataframe.sample(
            n=sample_size - len(sampled_dataframe),
            random_state=random_seed,
        )
        sampled_dataframe = pd.concat([sampled_dataframe, additional_dataframe], ignore_index=True)

    return sampled_dataframe.sort_values("query_id").reset_index(drop=True)


query_dataframe = pd.read_csv(QUERY_SET_PATH)
if "query_id" not in query_dataframe.columns or "query_text" not in query_dataframe.columns:
    raise ValueError("Query set must contain query_id and query_text columns.")

available_query_ids = set(blinded_dataframe["query_id"].unique())
query_dataframe = query_dataframe.loc[query_dataframe["query_id"].astype(int).isin(available_query_ids)].copy()

sampled_queries = sample_queries(
    query_dataframe=query_dataframe,
    sample_size=HUMAN_VALIDATION_QUERY_COUNT,
    random_seed=HUMAN_VALIDATION_RANDOM_SEED,
    stratify_column="type_of_food",
)
sampled_query_ids = set(sampled_queries["query_id"].astype(int).tolist())

print("Sampled query count:", len(sampled_queries))
print("Sampled query IDs:", sorted(sampled_query_ids))
if "type_of_food" in sampled_queries.columns:
    print("\nSampled type_of_food distribution:")
    print(sampled_queries["type_of_food"].value_counts(dropna=False))

sampled_queries.head()


Sampled query count: 40
Sampled query IDs: [15, 18, 21, 26, 27, 32, 54, 60, 61, 81, 90, 91, 94, 122, 142, 150, 163, 168, 170, 178, 198, 214, 232, 234, 284, 302, 306, 309, 339, 384, 390, 396, 406, 413, 421, 430, 450, 459, 467, 469]

Sampled type_of_food distribution:
type_of_food
Món bánh                      5
Ăn vặt                        2
Món hấp                       2
Món nước                      1
Món từ gà                     1
Các loại bánh                 1
Món khai vị                   1
Món nướng                     1
Nhanh và dễ                   1
Món ngon theo vùng miền       1
Món chính                     1
Bữa sáng đơn giản             1
Món ăn sáng                   1
Món kho                       1
Món kem                       1
Thức uống                     1
Món tráng miệng, giải khát    1
Món xào                       1
Món ngon ngày lạnh            1
Món chay                      1
Thực đơn hàng ngày            1
Ngày lễ Tết                   1
Món chè         

,query_id,query_text,source_doc_id,type_of_food
0,15,Bánh canh thịt bò ngon lạ miệng hấp dẫn bổ dưỡ...,4303,Món nước
1,18,Cà ri gà sữa tươi chuẩn vị nhà hàng đơn giản n...,8484,Món từ gà
2,21,"Mứt nghệ tươi vàng óng, thơm ngon, đơn giản ch...",8991,Ăn vặt
3,26,Bánh khoai tây nhân phô mai,734,Các loại bánh
4,27,Súp gà,922,Món khai vị


## 4. Build Annotator-Facing Table

The annotation table intentionally hides model information. `_doc_id_internal` is included only for later merging and will be hidden in Excel. The annotator should fill only `human_relevance` and `human_notes`.


In [4]:
def build_annotation_dataframe(
    blinded_dataframe: pd.DataFrame,
    sampled_query_ids: set[int],
) -> pd.DataFrame:
    """Build the Excel annotation sheet rows."""
    selected_dataframe = blinded_dataframe.loc[
        blinded_dataframe["query_id"].isin(sampled_query_ids)
    ].copy()
    selected_dataframe = selected_dataframe.sort_values(
        ["query_id", "blinded_position"],
        ascending=[True, True],
    ).reset_index(drop=True)

    annotation_dataframe = pd.DataFrame(
        {
            "query_id": selected_dataframe["query_id"].astype(int),
            "query_text": selected_dataframe["query_text"].astype(str),
            "candidate_id": selected_dataframe["candidate_id"].astype(str),
            "_doc_id_internal": selected_dataframe["doc_id"].astype(int),
            "title": selected_dataframe["recipe_title"].map(lambda value: compact_text(value, 220)),
            "recipe_type": selected_dataframe["recipe_type"].map(lambda value: compact_text(value, 80)),
            "description": selected_dataframe["recipe_description"].map(lambda value: compact_text(value, 900)),
            "human_relevance": "",
            "human_notes": "",
        }
    )
    return annotation_dataframe


annotation_dataframe = build_annotation_dataframe(
    blinded_dataframe=blinded_dataframe,
    sampled_query_ids=sampled_query_ids,
)

print("Annotation rows:", len(annotation_dataframe))
print("Rows per sampled query:")
print(annotation_dataframe.groupby("query_id").size().describe())
annotation_dataframe.head()


Annotation rows: 2000
Rows per sampled query:
count    40.0
mean     50.0
std       0.0
min      50.0
25%      50.0
50%      50.0
75%      50.0
max      50.0
dtype: float64


,query_id,query_text,candidate_id,_doc_id_internal,title,recipe_type,description,human_relevance,human_notes
0,15,Bánh canh thịt bò ngon lạ miệng hấp dẫn bổ dưỡ...,D001,2978,"Bánh ít trần nhân tôm thịt mềm dẻo, thơm ngon ...",Món bánh,"Đối với ẩm thực Huế, bánh ít trần nhân tôm thị...",,
1,15,Bánh canh thịt bò ngon lạ miệng hấp dẫn bổ dưỡ...,D002,5362,"Rau lang xào thịt bò cực ngon, cực đơn giản ch...",Món xào,Rau lang là một trong những loại thực phẩm thơ...,,
2,15,Bánh canh thịt bò ngon lạ miệng hấp dẫn bổ dưỡ...,D003,1728,Canh thịt bò lá lốt thơm ngon dễ làm bổ dưỡng,Món canh,Canh thịt bò lá lốt là món ăn vô cùng thơm ngo...,,
3,15,Bánh canh thịt bò ngon lạ miệng hấp dẫn bổ dưỡ...,D004,5537,"Đậu rồng xào thịt bò thơm ngon, lạ miệng, cực ...",Món xào,Đậu rồng mang hương vị giòn ngon đặc trưng và ...,,
4,15,Bánh canh thịt bò ngon lạ miệng hấp dẫn bổ dưỡ...,D005,4300,"Bánh canh tôm, thịt bằm nước cốt dừa đúng chuẩ...",Món nước,Bánh canh bột gạo nước cốt dừa là một món ăn q...,,


## 5. Export Protected Excel Workbook

The workbook uses data validation to restrict `human_relevance` to `0`, `1`, `2`, or `3`. The internal doc ID column is hidden but preserved for later merging. The sheet is intentionally not protected, because protection can make annotation awkward in Excel; the dropdown validation is the main guard against invalid labels.


In [5]:
RUBRIC_ROWS = [
    {
        "label": 3,
        "name": "Highly relevant",
        "definition": "Directly satisfies the query intent or is an excellent recipe match.",
    },
    {
        "label": 2,
        "name": "Relevant",
        "definition": "Close variant or reasonable substitute that remains useful for the query.",
    },
    {
        "label": 1,
        "name": "Somewhat relevant",
        "definition": "Weakly related; useful only for broad exploration or partial support.",
    },
    {
        "label": 0,
        "name": "Not relevant",
        "definition": "Does not substantially help answer the query.",
    },
]


def export_annotation_workbook(
    annotation_dataframe: pd.DataFrame,
    sampled_queries: pd.DataFrame,
    workbook_path: Path,
) -> None:
    """Export a user-friendly Excel workbook for human annotation."""
    workbook_path.parent.mkdir(parents=True, exist_ok=True)
    rubric_dataframe = pd.DataFrame(RUBRIC_ROWS)

    with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
        rubric_dataframe.to_excel(writer, sheet_name="rubric", index=False)
        annotation_dataframe.to_excel(writer, sheet_name="annotation", index=False)
        sampled_queries.to_excel(writer, sheet_name="sampled_queries", index=False)

    workbook = load_workbook(workbook_path)
    annotation_sheet = workbook["annotation"]
    rubric_sheet = workbook["rubric"]
    sampled_sheet = workbook["sampled_queries"]

    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(color="FFFFFF", bold=True)
    input_fill = PatternFill("solid", fgColor="FFF2CC")
    warning_fill = PatternFill("solid", fgColor="F8CBAD")
    query_fill_a = PatternFill("solid", fgColor="FFFFFF")
    query_fill_b = PatternFill("solid", fgColor="EAF2F8")

    for worksheet in [annotation_sheet, rubric_sheet, sampled_sheet]:
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions
        for cell in worksheet[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    annotation_columns = {
        cell.value: cell.column
        for cell in annotation_sheet[1]
    }
    human_relevance_column = annotation_columns["human_relevance"]
    human_notes_column = annotation_columns["human_notes"]
    internal_doc_id_column = annotation_columns["_doc_id_internal"]

    annotation_sheet.column_dimensions[get_column_letter(internal_doc_id_column)].hidden = True
    annotation_sheet.freeze_panes = "A2"

    column_widths = {
        "A": 10,
        "B": 36,
        "C": 12,
        "D": 14,
        "E": 36,
        "F": 16,
        "G": 70,
        "H": 18,
        "I": 36,
    }
    for column_letter, width in column_widths.items():
        annotation_sheet.column_dimensions[column_letter].width = width

    for row_index in range(2, annotation_sheet.max_row + 1):
        query_id = annotation_sheet.cell(row=row_index, column=annotation_columns["query_id"]).value
        row_fill = query_fill_a if int(query_id) % 2 == 0 else query_fill_b
        for column_index in range(1, annotation_sheet.max_column + 1):
            cell = annotation_sheet.cell(row=row_index, column=column_index)
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            cell.fill = row_fill

        relevance_cell = annotation_sheet.cell(row=row_index, column=human_relevance_column)
        notes_cell = annotation_sheet.cell(row=row_index, column=human_notes_column)
        relevance_cell.fill = input_fill
        notes_cell.fill = input_fill

    relevance_column_letter = get_column_letter(human_relevance_column)
    relevance_range = f"{relevance_column_letter}2:{relevance_column_letter}{annotation_sheet.max_row}"
    data_validation = DataValidation(
        type="list",
        formula1='"0,1,2,3"',
        allow_blank=False,
        showErrorMessage=True,
        errorTitle="Invalid relevance label",
        error="Please select one of: 0, 1, 2, 3.",
        promptTitle="Human relevance",
        prompt="Select 0, 1, 2, or 3 according to the rubric sheet.",
    )
    annotation_sheet.add_data_validation(data_validation)
    data_validation.add(relevance_range)

    missing_label_rule = FormulaRule(
        formula=[f'ISBLANK(${relevance_column_letter}2)'],
        fill=warning_fill,
    )
    annotation_sheet.conditional_formatting.add(relevance_range, missing_label_rule)

    annotation_sheet["H1"].comment = Comment(
        "Only this column and human_notes are editable. Use 0, 1, 2, or 3 only.",
        "Codex",
    )
    for worksheet in [rubric_sheet, sampled_sheet]:
        for column_cells in worksheet.columns:
            column_letter = get_column_letter(column_cells[0].column)
            max_length = max(len(str(cell.value or "")) for cell in column_cells)
            worksheet.column_dimensions[column_letter].width = min(max(max_length + 2, 12), 80)
            for cell in column_cells:
                cell.alignment = Alignment(vertical="top", wrap_text=True)

    workbook.save(workbook_path)


export_annotation_workbook(
    annotation_dataframe=annotation_dataframe,
    sampled_queries=sampled_queries,
    workbook_path=ANNOTATION_WORKBOOK_PATH,
)

print("Saved human validation workbook to:", ANNOTATION_WORKBOOK_PATH)


Saved human validation workbook to: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\human_validation\human_validation_annotation_sheet.xlsx


## 6. Read Back the Completed Workbook

After annotation is finished, save a copy as:

```text
groundtruth_outputs/human_validation/human_validation_annotation_sheet_completed.xlsx
```

Then run this section to validate the human input and export a normalized CSV that `3_validate_groundtruth.ipynb` can read.


In [6]:
def load_completed_human_workbook(completed_workbook_path: Path) -> pd.DataFrame:
    """Load and validate the completed human annotation workbook."""
    if not completed_workbook_path.exists():
        raise FileNotFoundError(
            "Completed workbook not found. Ask the annotator to save the filled file as: "
            f"{completed_workbook_path}"
        )

    completed_dataframe = pd.read_excel(completed_workbook_path, sheet_name="annotation")
    required_columns = {
        "query_id",
        "query_text",
        "candidate_id",
        "_doc_id_internal",
        "human_relevance",
        "human_notes",
    }
    missing_columns = required_columns - set(completed_dataframe.columns)
    if missing_columns:
        raise ValueError(f"Completed workbook is missing columns: {sorted(missing_columns)}")

    completed_dataframe = completed_dataframe.copy()
    completed_dataframe["query_id"] = completed_dataframe["query_id"].astype(int)
    completed_dataframe["_doc_id_internal"] = completed_dataframe["_doc_id_internal"].astype(int)

    missing_relevance_count = completed_dataframe["human_relevance"].isna().sum()
    if missing_relevance_count:
        raise ValueError(f"There are {missing_relevance_count} rows without human_relevance.")

    completed_dataframe["human_relevance"] = completed_dataframe["human_relevance"].astype(int)
    invalid_values = sorted(set(completed_dataframe["human_relevance"]) - {0, 1, 2, 3})
    if invalid_values:
        raise ValueError(f"Invalid human_relevance values found: {invalid_values}")

    duplicate_count = completed_dataframe.duplicated(subset=["query_id", "_doc_id_internal"]).sum()
    if duplicate_count:
        raise ValueError(f"Found {duplicate_count} duplicate query-document rows.")

    normalized_dataframe = completed_dataframe.rename(columns={"_doc_id_internal": "doc_id"})
    output_columns = [
        "query_id",
        "doc_id",
        "candidate_id",
        "query_text",
        "title",
        "recipe_type",
        "description",
        "human_relevance",
        "human_notes",
    ]
    return normalized_dataframe[[column for column in output_columns if column in normalized_dataframe.columns]]


# Run this after the completed workbook exists.
# completed_human_dataframe = load_completed_human_workbook(COMPLETED_WORKBOOK_PATH)
# completed_human_dataframe.to_csv(COMPLETED_CSV_PATH, index=False, encoding="utf-8-sig")
# print("Saved normalized completed CSV to:", COMPLETED_CSV_PATH)
# completed_human_dataframe.head()
